## **GET METADATA FROM DATAHUB AND CONVERT INTO JSONL**

In [13]:
# 01_export_columns.py
import os
import json
import requests
import re
from typing import Dict, List, Any, Optional
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

# ===== Config =====
GMS_BASE = os.getenv("DATAHUB_GMS_URL", "http://datahub-gms:8080")
GRAPHQL_URL = os.getenv("DATAHUB_GRAPHQL_URL", f"{GMS_BASE}/api/graphql")
TOKEN = os.getenv("DATAHUB_TOKEN")
if not TOKEN:
    raise RuntimeError("DATAHUB_TOKEN is not set.")
PAGE_SIZE = int(os.getenv("DATAHUB_PAGE_SIZE", "100"))
TIMEOUT = 60

# Output directory for per-database JSONL files
OUTPUT_DIR = os.getenv("COLUMNS_OUTPUT_DIR", "./exports")

# Platforms using database.schema.table
RELATIONAL_3LEVEL = {"postgres", "snowflake", "redshift", "bigquery", "mssql", "mysql"}

# ===== GraphQL =====
SCROLL_QUERY = """
query scrollPG($types: [EntityType!], $query: String!, $count: Int!, $scrollId: String) {
  scrollAcrossEntities(input: { types: $types, query: $query, count: $count, scrollId: $scrollId }) {
    nextScrollId
    searchResults {
      entity {
        urn
        type
        ... on Dataset { urn }
      }
    }
  }
}
""".strip()

DATASET_META_QUERY = """
query datasetMeta($urn: String!) {
  dataset(urn: $urn) {
    urn
    platform { name properties { datasetNameDelimiter } }
    parentContainers {
      containers {
        properties { name description }
        editableProperties { description }   # <-- pick up UI/GraphQL-edited docs
      }
    }
    schemaMetadata(version: 0) {
      fields { fieldPath description nativeDataType type }
    }
  }
}
""".strip()

def _graphql(query: str, variables: dict) -> dict:
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"}
    resp = requests.post(GRAPHQL_URL, json={"query": query, "variables": variables},
                         headers=headers, timeout=TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    if data.get("errors"):
        raise RuntimeError(data["errors"][0])
    return data["data"]

def iter_dataset_urns():
    scroll_id = None
    while True:
        data = _graphql(
            SCROLL_QUERY,
            {"types": ["DATASET"], "query": "*", "count": PAGE_SIZE, "scrollId": scroll_id},
        )["scrollAcrossEntities"]
        for item in data.get("searchResults") or []:
            ent = item.get("entity") or {}
            urn = ent.get("urn")
            if urn:
                yield urn
        scroll_id = data.get("nextScrollId")
        if not scroll_id:
            break

def _parse_urn_name(urn: str) -> Optional[str]:
    # urn:li:dataset:(platformUrn,name,origin) -> name
    try:
        inner = urn[urn.index("(")+1 : urn.rindex(")")]
        _, name_part, _ = inner.split(",", 2)
        return name_part
    except Exception:
        return None

def _split_from_urn(urn: str, platform_name: Optional[str], delim: str = "."):
    # Returns (full_name, database, schema, table) from URN
    name_part = _parse_urn_name(urn)
    if not name_part:
        return None, None, None, None
    plat = (platform_name or "").lower()
    if plat in RELATIONAL_3LEVEL:
        parts = name_part.split(delim)
        if len(parts) >= 3:
            return name_part, parts[0], parts[1], delim.join(parts[2:])
        elif len(parts) == 2:
            return name_part, parts[0], None, parts[1]
        else:
            return name_part, None, None, parts[0]
    return name_part, None, None, name_part

def fetch_dataset_columns(urn: str) -> List[Dict[str, Any]]:
    """
    Returns a list of column dicts:
      {database, table, column, datatype, description, database_description}
      (and includes {schema} ONLY for platforms where it makes sense and is present)
    """
    d = _graphql(DATASET_META_QUERY, {"urn": urn}).get("dataset")
    if not d:
        return []

    platform = d.get("platform") or {}
    platform_name = (platform.get("name") or "").lower()
    delim = (platform.get("properties") or {}).get("datasetNameDelimiter") or "."

    _, db, schema, table = _split_from_urn(urn, platform_name, delim)

    # Find database description from parent containers
    db_desc = ""
    containers = ((d.get("parentContainers") or {}).get("containers") or [])
    for c in containers:
        props = c.get("properties") or {}
        edit  = c.get("editableProperties") or {}
        if (props.get("name") or "").lower() == (db or "").lower():
            db_desc = (edit.get("description") or props.get("description") or "").strip()
            break

    # Optional fallback: first non-empty description up the chain
    if not db_desc:
        for c in containers:
            props = c.get("properties") or {}
            edit  = c.get("editableProperties") or {}
            cand = (edit.get("description") or props.get("description") or "").strip()
            if cand:
                db_desc = cand
                break

    # MySQL: schema concept == database; omit "schema" from the output entirely
    omit_schema = (platform_name == "mysql")

    rows = []
    sm = d.get("schemaMetadata") or {}
    for f in (sm.get("fields") or []):
        row = {
            "database": db,
            "table": table,
            "column": f.get("fieldPath"),
            "datatype": f.get("nativeDataType") or (f.get("type") or ""),
            "description": f.get("description") or "",
            "database_description": db_desc,
        }
        # Include schema only when it exists and we're not on MySQL
        if (schema is not None) and (not omit_schema):
            row["schema"] = schema
        rows.append(row)
    return rows

def _sanitize_filename(name: str) -> str:
    """
    Make a safe filename from database name.
    - Lowercase
    - Replace non-alphanum with underscores
    - Collapse multiple underscores
    """
    if not name:
        return "unknown"
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9\\-_.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Keep one open file handle per database
    writers: Dict[str, Any] = {}
    counts: Dict[str, int] = {}

    def get_writer(db_name: Optional[str]):
        db_key = db_name or "unknown"
        safe = _sanitize_filename(db_key)
        path = os.path.join(OUTPUT_DIR, f"{safe}.jsonl")
        if safe not in writers:
            writers[safe] = open(path, "w", encoding="utf-8")
            counts[safe] = 0
        return writers[safe], safe

    try:
        for urn in iter_dataset_urns():
            try:
                rows = fetch_dataset_columns(urn)
                for row in rows:
                    writer, key = get_writer(row.get("database"))
                    writer.write(json.dumps(row, ensure_ascii=False) + "\n")
                    counts[key] += 1
            except Exception as e:
                # If dataset fails, write to a generic 'unknown.jsonl'
                writer, key = get_writer("unknown")
                writer.write(json.dumps({"error": f"{urn}: {e}"}, ensure_ascii=False) + "\n")
                counts[key] += 1
    finally:
        for fh in writers.values():
            try:
                fh.close()
            except Exception:
                pass

    # Summary
    print(f"Exported per database to: {os.path.abspath(OUTPUT_DIR)}")
    for db_file, n in sorted(counts.items()):
        print(f"- {db_file}.jsonl: {n} rows")

if __name__ == "__main__":
    main()


Exported per database to: /app/datahub_metadata
- classicmodels.jsonl: 59 rows
